In [16]:
# Runtime radius mode model (2-stage: NO_RADIUS / LOCAL vs MID)
# Vstup: query, orders_sum, deal_region_res4, bucket_label_km
# Filtr: (query, region) s total_orders < MIN_TOTAL_ORDERS se zahodí hned na začátku

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
import joblib

# --- Config ---
INPUT_CSV = "data/train_new.csv"
MODEL_NAME = "all-MiniLM-L6-v2"
DEVICE = "cpu"
MIN_TOTAL_ORDERS = 6   # filtr vstupu: řádky pod tímto se zahodí
Q_TARGET = 0.80
RANDOM_STATE = 42
TEST_SIZE = 0.2
RADIUS_LOCAL_KM = 64
RADIUS_MID_KM = 256
NO_RADIUS_DEFAULT = None
NO_RADIUS_THRESHOLD = 0.45   # default: lepší recall, slušná precision
MID_THRESHOLD = 0.45   # Stage 2: MID když p_mid >= 0.45 (ne 0.50)
OUT_STAGE1 = "model_stage1_noradius.joblib"
OUT_STAGE2 = "model_stage2_local_mid.joblib"
# Šedá zóna: p_nr in (0.40, 0.60) → UNKNOWN → fallback (goods token → NO_RADIUS, jinak MID)
USE_GRAY_ZONE = False
NO_RADIUS_HIGH = 0.60
NO_RADIUS_LOW = 0.40
GOODS_TOKENS = ("windows", "office", "license")

bucket_order = {
    "0–2": 0, "2–4": 1, "4–8": 2, "8–16": 3, "16–32": 4,
    "32–64": 5, "64–128": 6, "128–256": 7, "256+": 8,
}


class SentenceTransformerWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, model_name=MODEL_NAME, device=DEVICE):
        self.model_name = model_name
        self.device = device
        self.model = None
    def fit(self, X, y=None):
        if self.model is None:
            self.model = SentenceTransformer(self.model_name, device=self.device)
        return self
    def transform(self, X):
        texts = X.tolist() if isinstance(X, pd.Series) else list(X)
        return self.model.encode(texts, show_progress_bar=False)


def make_pipeline():
    preprocess = ColumnTransformer(
        transformers=[
            ("text", SentenceTransformerWrapper(model_name=MODEL_NAME, device=DEVICE), "query_text"),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["deal_region_res4"]),
        ],
        remainder="drop",
    )
    return Pipeline(steps=[("prep", preprocess), ("clf", LogisticRegression(max_iter=2000, n_jobs=-1))])


# --- Načtení + filtr MIN_TOTAL_ORDERS (ostatní řádky zahodit) + Q80 ---
df = pd.read_csv(INPUT_CSV).rename(columns={"query": "query_text"})
n_rows_file = len(df)
print(f"Řádků v souboru {INPUT_CSV}: {n_rows_file}")
required = {"query_text", "orders_sum", "deal_region_res4", "bucket_label_km"}
if required - set(df.columns):
    raise ValueError(f"Chybí sloupce: {required - set(df.columns)}")

df["query_text"] = df["query_text"].astype(str).str.strip()
df = df[(df["query_text"] != "") & (df["query_text"].str.len() >= 2)]
df = df[df["bucket_label_km"].isin(bucket_order)].copy()
df["bucket_id"] = df["bucket_label_km"].map(bucket_order).astype(int)
df["orders_sum"] = pd.to_numeric(df["orders_sum"], errors="coerce").fillna(0).astype(float)

# Agregace po (query, region, bucket)
g = df.groupby(["query_text", "deal_region_res4", "bucket_id", "bucket_label_km"], as_index=False).agg(orders_sum=("orders_sum", "sum"))
tot = g.groupby(["query_text", "deal_region_res4"], as_index=False).agg(total_orders=("orders_sum", "sum"))

# Filtr: pouze (query, region) s total_orders >= MIN_TOTAL_ORDERS; ostatní zahodit
keep = tot[tot["total_orders"] >= MIN_TOTAL_ORDERS][["query_text", "deal_region_res4"]]
g = g.merge(keep, on=["query_text", "deal_region_res4"], how="inner")
tot = tot.merge(keep, on=["query_text", "deal_region_res4"], how="inner")

g = g.merge(tot, on=["query_text", "deal_region_res4"], how="left")
g["orders_share"] = g["orders_sum"] / g["total_orders"]
g = g.sort_values(["query_text", "deal_region_res4", "bucket_id"])
g["cum_share"] = g.groupby(["query_text", "deal_region_res4"])["orders_share"].cumsum()

# Q80 label per (query, region)
q = (
    g[g["cum_share"] >= Q_TARGET]
    .groupby(["query_text", "deal_region_res4"], as_index=False)
    .first()[["query_text", "deal_region_res4", "bucket_id", "bucket_label_km", "total_orders"]]
    .rename(columns={"bucket_id": "label_bucket_id", "bucket_label_km": "label_bucket_label_km"})
)
q["sample_weight"] = np.log1p(q["total_orders"])

print(f"Soubor: {n_rows_file} řádků  →  pro trénink: {len(q)} řádků")
print("Rozložení label_bucket_id:")
print(q["label_bucket_id"].value_counts().sort_index())



Řádků v souboru data/train_new.csv: 220062
Soubor: 220062 řádků  →  pro trénink: 9061 řádků
Rozložení label_bucket_id:
label_bucket_id
0       1
1       3
2      38
3     363
4    1672
5    2151
6    1136
7     795
8    2902
Name: count, dtype: int64


In [17]:
# Stage 1: NO_RADIUS (binary) — bucket_id == 8 (256+)
q["label_noradius"] = (q["label_bucket_id"] == 8).astype(int)
X1 = q[["query_text", "deal_region_res4"]]
y1 = q["label_noradius"].astype(int)
w1 = q["sample_weight"].astype(float)

X1_train, X1_val, y1_train, y1_val, w1_train, w1_val = train_test_split(
    X1, y1, w1, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y1
)
model1 = make_pipeline()
print("Trénování Stage 1 (NO_RADIUS)...")
model1.fit(X1_train, y1_train, clf__sample_weight=w1_train)
# Evaluace přes predict_proba + threshold (stejné pravidlo jako runtime)
p1 = model1.predict_proba(X1_val)[:, 1]
y1_pred = (p1 >= NO_RADIUS_THRESHOLD).astype(int)
print(f"Threshold NO_RADIUS: {NO_RADIUS_THRESHOLD}")
print("\nStage 1 validace:")
print(classification_report(y1_val, y1_pred))
print(confusion_matrix(y1_val, y1_pred))

# Porovnání thresholdů (zkus 0.45 / 0.40 pro posun recall/precision):
for th in [NO_RADIUS_THRESHOLD, 0.55, 0.45, 0.40]:
    pred = (p1 >= th).astype(int)
    r = classification_report(y1_val, pred, output_dict=True, zero_division=0)
    print(f"  th={th}: P={r['1']['precision']:.3f} R={r['1']['recall']:.3f} F1={r['1']['f1-score']:.3f}")



Trénování Stage 1 (NO_RADIUS)...


/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the envir

Threshold NO_RADIUS: 0.45

Stage 1 validace:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1232
           1       0.75      0.64      0.69       581

    accuracy                           0.82      1813
   macro avg       0.80      0.77      0.78      1813
weighted avg       0.81      0.82      0.81      1813

[[1109  123]
 [ 211  370]]
  th=0.45: P=0.751 R=0.637 F1=0.689
  th=0.55: P=0.816 R=0.549 F1=0.656
  th=0.45: P=0.751 R=0.637 F1=0.689
  th=0.4: P=0.716 R=0.687 F1=0.701


In [18]:
# Stage 2: LOCAL vs MID (binary) — pouze bucket_id 0..7; LOCAL=0..5, MID=6..7
q2 = q[q["label_bucket_id"] <= 7].copy()
q2["label_local_mid"] = (q2["label_bucket_id"] >= 6).astype(int)
X2 = q2[["query_text", "deal_region_res4"]]
y2 = q2["label_local_mid"].astype(int)
w2 = q2["sample_weight"].astype(float)

X2_train, X2_val, y2_train, y2_val, w2_train, w2_val = train_test_split(
    X2, y2, w2, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y2
)
model2 = make_pipeline()
print("Trénování Stage 2 (LOCAL vs MID)...")
model2.fit(X2_train, y2_train, clf__sample_weight=w2_train)
p2 = model2.predict_proba(X2_val)[:, 1]
y2_pred = (p2 >= MID_THRESHOLD).astype(int)
print(f"Threshold MID: {MID_THRESHOLD}")
print("\nStage 2 validace:")
print(classification_report(y2_val, y2_pred))
print(confusion_matrix(y2_val, y2_pred))
for th in [MID_THRESHOLD, 0.50, 0.40, 0.35]:
    pred = (p2 >= th).astype(int)
    r = classification_report(y2_val, pred, output_dict=True, zero_division=0)
    print(f"  th={th}: P={r['1']['precision']:.3f} R={r['1']['recall']:.3f} F1={r['1']['f1-score']:.3f}")



Trénování Stage 2 (LOCAL vs MID)...


/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Threshold MID: 0.45

Stage 2 validace:
              precision    recall  f1-score   support

           0       0.84      0.86      0.85       846
           1       0.67      0.63      0.65       386

    accuracy                           0.79      1232
   macro avg       0.75      0.75      0.75      1232
weighted avg       0.78      0.79      0.79      1232

[[725 121]
 [141 245]]
  th=0.45: P=0.669 R=0.635 F1=0.652
  th=0.5: P=0.692 R=0.575 F1=0.628
  th=0.4: P=0.645 R=0.681 F1=0.662
  th=0.35: P=0.597 R=0.715 F1=0.651


In [19]:
# Uložení modelů + runtime helper
joblib.dump({
    "model": model1,
    "bucket_order": bucket_order,
    "q_target": Q_TARGET,
    "min_total_orders": MIN_TOTAL_ORDERS,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "label": "NO_RADIUS (Q80 bucket_id == 8)",
    "runtime_threshold": NO_RADIUS_THRESHOLD,
}, OUT_STAGE1)
joblib.dump({
    "model": model2,
    "bucket_order": bucket_order,
    "q_target": Q_TARGET,
    "min_total_orders": MIN_TOTAL_ORDERS,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "label": "LOCAL vs MID (bucket_id 0..7)",
    "runtime_radius_km": {"LOCAL": RADIUS_LOCAL_KM, "MID": RADIUS_MID_KM, "NO_RADIUS": NO_RADIUS_DEFAULT},
    "runtime_mid_threshold": MID_THRESHOLD,
}, OUT_STAGE2)
print(f"Uloženo: {OUT_STAGE1}, {OUT_STAGE2}")


def _query_has_goods_token(query_text: str) -> bool:
    q = str(query_text).lower()
    return any(t in q for t in GOODS_TOKENS)


def predict_mode(query_text: str, deal_region_res4: str, no_radius_threshold: float = NO_RADIUS_THRESHOLD, mid_threshold: float = MID_THRESHOLD):
    X = pd.DataFrame({"query_text": [str(query_text)], "deal_region_res4": [str(deal_region_res4)]})
    p_nr = float(model1.predict_proba(X)[0, 1])
    p_mid = float(model2.predict_proba(X)[0, 1])

    if USE_GRAY_ZONE:
        if p_nr >= NO_RADIUS_HIGH:
            return {"mode": "NO_RADIUS", "radius_km": NO_RADIUS_DEFAULT, "p_no_radius": p_nr}
        if p_nr <= NO_RADIUS_LOW:
            pass  # → Stage 2 (LOCAL vs MID)
        else:
            if _query_has_goods_token(query_text):
                return {"mode": "NO_RADIUS", "radius_km": NO_RADIUS_DEFAULT, "p_no_radius": p_nr}
            return {"mode": "MID", "radius_km": RADIUS_MID_KM, "p_no_radius": p_nr, "p_mid": p_mid}
    else:
        if p_nr >= no_radius_threshold:
            return {"mode": "NO_RADIUS", "radius_km": NO_RADIUS_DEFAULT, "p_no_radius": p_nr}

    if p_mid >= mid_threshold:
        return {"mode": "MID", "radius_km": RADIUS_MID_KM, "p_no_radius": p_nr, "p_mid": p_mid}
    return {"mode": "LOCAL", "radius_km": RADIUS_LOCAL_KM, "p_no_radius": p_nr, "p_mid": p_mid}



Uloženo: model_stage1_noradius.joblib, model_stage2_local_mid.joblib


## Test the model

Enter GPS coordinates (lat, lon) and query. GPS is converted to H3 (res 4) and the model returns mode (LOCAL / MID / NO_RADIUS), radius in km and probabilities.

In [25]:
# Test model: GPS -> H3 res4 + query -> predict_mode
import h3

H3_RES = 4  # deal_region_res4

def gps_to_h3_res4(gps_str):
    """Convert 'lat, lon' string to H3 index resolution 4."""
    parts = [p.strip() for p in gps_str.split(",")]
    if len(parts) != 2:
        raise ValueError("GPS format: lat, lon (e.g. 41.85322835386527, -87.64094972748221)")
    lat, lon = float(parts[0]), float(parts[1])
    return h3.latlng_to_cell(lat, lon, H3_RES)

def test_model(gps_str, query_text, no_radius_threshold=NO_RADIUS_THRESHOLD):
    """Return all model outputs for given GPS and query."""
    deal_region_res4 = gps_to_h3_res4(gps_str)
    result = predict_mode(query_text, deal_region_res4, no_radius_threshold=no_radius_threshold)
    return {
        "gps": gps_str,
        "query": query_text,
        "deal_region_res4": deal_region_res4,
        **result,
    }

# --- Test: sběr výsledků do tabulky se společným záhlavím ---
GPS_STR = "41.85322835386527, -87.64094972748221"

test_queries = [
    "universal", "massage", "masage", "facial", "smog check", "oil change",
    "trampoline park", "escape room", "zoo", "sky zone", "bowling", "seaworld",
    "great wolf lodge", "great wolf", "citypass", "whale watching", "hotel", "hotl",
    "windows 11", "microsoft office", "costco", "cosco", "windows 10", "airport parking",
    "architecture boat tour", "bared monkey", "bible museum", "big air",
    "big air trampoline park", "laser hair removal", "apple", "apple store", "brazilian",
    "botox", "botox injection", "botox treatment", "amc", "amc theaters", "amc movie",
    "sams club", "sams", "sams club membership", "sams club membership card", "sams club membership card",
]

rows = []
for query_text in test_queries:
    out = test_model(GPS_STR, query_text)
    rows.append({
        "query": out["query"],
        "mode": out["mode"],
        "radius_km": out["radius_km"],
        "p_no_radius": round(out["p_no_radius"], 4),
        "p_mid": round(out["p_mid"], 4) if out.get("p_mid") is not None else None,
    })

result_table = pd.DataFrame(rows)
result_table

,query,mode,radius_km,p_no_radius,p_mid
0,universal,NO_RADIUS,NaN,0.7872,NaN
1,massage,LOCAL,64.0,0.1496,0.1367
2,masage,LOCAL,64.0,0.2651,0.0804
3,facial,LOCAL,64.0,0.1622,0.0720
4,smog check,LOCAL,64.0,0.0745,0.0899
5,oil change,LOCAL,64.0,0.1001,0.2216
6,trampoline park,LOCAL,64.0,0.1426,0.2066
7,escape room,LOCAL,64.0,0.1153,0.1146
8,zoo,LOCAL,64.0,0.2672,0.4025
9,sky zone,LOCAL,64.0,0.1143,0.2183
